# External Features — Live Source Verification

Testing whether we can programmatically fetch the 3 external features:
1. DVOL (BTC & ETH) from CryptoDataDownload
2. Fear & Greed Index from alternative.me
3. News Sentiment from FRBSF

In [1]:
import pandas as pd
import requests
from io import StringIO

## 1. DVOL — CryptoDataDownload CSVs

Sources:
- https://www.cryptodatadownload.com/cdd/DeriBit_volatility_OHLC_BTC.csv
- https://www.cryptodatadownload.com/cdd/DeriBit_volatility_OHLC_ETH.csv

In [2]:
# BTC DVOL
url_btc = "https://www.cryptodatadownload.com/cdd/DeriBit_volatility_OHLC_BTC.csv"
resp_btc = requests.get(url_btc, timeout=30)
print(f"Status: {resp_btc.status_code}")
print(f"Content-Type: {resp_btc.headers.get('Content-Type')}")
print(f"First 500 chars:\n{resp_btc.text[:500]}")

Status: 200
Content-Type: text/csv
First 500 chars:
https://www.CryptoDataDownload.com
date,symbol,open,high,low,close
2026-05-09,BTC,37.66,37.72,37.49,37.54
2026-05-08,BTC,38.85,39.03,37.42,37.66
2026-05-07,BTC,39.41,39.41,38.25,38.85
2026-05-06,BTC,39.4,40.54,39.09,39.41
2026-05-05,BTC,41.0,41.16,38.86,40.28
2026-05-04,BTC,39.05,41.12,38.86,41.0
2026-05-03,BTC,39.31,40.92,38.65,39.05
2026-05-02,BTC,38.52,39.31,38.51,39.31
2026-05-01,BTC,38.88,39.3,37.97,38.52
2026-04-30,BTC,39.62,40.06,38.86,38.88
2026-04-29,BTC,40.58,40.71,39.46,39.62
2026-04-


In [3]:
# Parse BTC DVOL — skip CryptoDataDownload header line if present
lines = resp_btc.text.split('\n')
skip = 1 if 'cryptodatadownload' in lines[0].lower() else 0
df_btc_dvol = pd.read_csv(StringIO('\n'.join(lines[skip:])))
print(f"Shape: {df_btc_dvol.shape}")
print(f"Columns: {list(df_btc_dvol.columns)}")
print(f"\nFirst 5 rows:")
display(df_btc_dvol.head())
print(f"\nLast 5 rows:")
display(df_btc_dvol.tail())

Shape: (1873, 6)
Columns: ['date', 'symbol', 'open', 'high', 'low', 'close']

First 5 rows:


,date,symbol,open,high,low,close
0,2026-05-09,BTC,37.66,37.72,37.49,37.54
1,2026-05-08,BTC,38.85,39.03,37.42,37.66
2,2026-05-07,BTC,39.41,39.41,38.25,38.85
3,2026-05-06,BTC,39.40,40.54,39.09,39.41
4,2026-05-05,BTC,41.00,41.16,38.86,40.28



Last 5 rows:


,date,symbol,open,high,low,close
1868,2021-03-28,BTC,79.69,81.28,78.20,79.07
1869,2021-03-27,BTC,84.52,84.60,77.81,79.69
1870,2021-03-26,BTC,89.74,89.74,80.22,84.52
1871,2021-03-25,BTC,95.03,97.43,87.25,89.74
1872,2021-03-24,BTC,84.80,95.94,80.52,95.04


In [4]:
# ETH DVOL
url_eth = "https://www.cryptodatadownload.com/cdd/DeriBit_volatility_OHLC_ETH.csv"
resp_eth = requests.get(url_eth, timeout=30)
print(f"Status: {resp_eth.status_code}")
lines = resp_eth.text.split('\n')
skip = 1 if 'cryptodatadownload' in lines[0].lower() else 0
df_eth_dvol = pd.read_csv(StringIO('\n'.join(lines[skip:])))
print(f"Shape: {df_eth_dvol.shape}")
print(f"\nLast 5 rows:")
display(df_eth_dvol.tail())

Status: 200
Shape: (1873, 6)

Last 5 rows:


,date,symbol,open,high,low,close
1868,2021-03-28,ETH,87.46,88.15,82.81,84.22
1869,2021-03-27,ETH,90.28,90.67,84.86,87.46
1870,2021-03-26,ETH,105.38,105.92,83.04,90.27
1871,2021-03-25,ETH,106.73,109.35,102.00,105.39
1872,2021-03-24,ETH,94.18,106.94,92.67,106.73


## 2. Fear & Greed Index — alternative.me

Source: https://alternative.me/crypto/fear-and-greed-index/

Testing if there's a usable API or if we need to scrape/download manually.

In [5]:
# Try the JSON API that exists at alternative.me
url_fng = "https://api.alternative.me/fng/?limit=5&format=json"
resp_fng = requests.get(url_fng, timeout=30)
print(f"Status: {resp_fng.status_code}")
print(f"Content-Type: {resp_fng.headers.get('Content-Type')}")
if resp_fng.status_code == 200:
    data = resp_fng.json()
    print(f"\nResponse keys: {list(data.keys())}")
    print(f"\nData (last 5 days):")
    for entry in data.get('data', []):
        ts = pd.Timestamp(int(entry['timestamp']), unit='s')
        print(f"  {ts.date()} -> value={entry['value']} ({entry['value_classification']})")
else:
    print(f"API failed. Response: {resp_fng.text[:300]}")

Status: 200
Content-Type: application/json

Response keys: ['name', 'data', 'metadata']

Data (last 5 days):
  2026-05-09 -> value=38 (Fear)
  2026-05-08 -> value=38 (Fear)
  2026-05-07 -> value=47 (Neutral)
  2026-05-06 -> value=46 (Fear)
  2026-05-05 -> value=50 (Neutral)


In [6]:
# Try fetching full history to see how far back it goes
url_fng_all = "https://api.alternative.me/fng/?limit=0&format=json"
resp_fng_all = requests.get(url_fng_all, timeout=30)
print(f"Status: {resp_fng_all.status_code}")
if resp_fng_all.status_code == 200:
    data_all = resp_fng_all.json()
    records = data_all.get('data', [])
    print(f"Total records: {len(records)}")
    if records:
        first_ts = pd.Timestamp(int(records[-1]['timestamp']), unit='s')
        last_ts = pd.Timestamp(int(records[0]['timestamp']), unit='s')
        print(f"Date range: {first_ts.date()} to {last_ts.date()}")
else:
    print(f"Failed: {resp_fng_all.text[:300]}")

Status: 200
Total records: 3016
Date range: 2018-02-01 to 2026-05-09


## 3. News Sentiment — FRBSF

Source: https://www.frbsf.org/wp-content/uploads/news_sentiment_data.xlsx

Testing if the xlsx is downloadable and what the update frequency really is.

In [7]:
# Try downloading the xlsx
url_sent = "https://www.frbsf.org/wp-content/uploads/news_sentiment_data.xlsx"
resp_sent = requests.get(url_sent, timeout=30)
print(f"Status: {resp_sent.status_code}")
print(f"Content-Type: {resp_sent.headers.get('Content-Type')}")
print(f"Content-Length: {len(resp_sent.content)} bytes")

Status: 200
Content-Type: application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
Content-Length: 414893 bytes


In [ ]:
# Parse the xlsx — read the "Data" sheet specifically
if resp_sent.status_code == 200:
    from io import BytesIO
    
    # Check available sheets
    xls = pd.ExcelFile(BytesIO(resp_sent.content))
    print(f"Available sheets: {xls.sheet_names}")
    
    # Read the Data sheet
    df_sent = pd.read_excel(BytesIO(resp_sent.content), sheet_name="Data")
    print(f"\nShape: {df_sent.shape}")
    print(f"Columns: {list(df_sent.columns)}")
    print(f"\nFirst 5 rows:")
    display(df_sent.head())
    print(f"\nLast 5 rows (most recent data):")
    display(df_sent.tail())
else:
    print("Download failed")

In [10]:
# Compare with local file to see how stale it is
local_sent = pd.read_csv('/home/pablo/Secondary-Model/src/Data_MLA/XFeatures/News_Sentiment_Data.csv')
print(f"Local file last date: {local_sent.iloc[-1, 0]}")
print(f"Local file shape: {local_sent.shape}")
if resp_sent.status_code == 200:
    print(f"Remote file last date: {df_sent.iloc[-1, 0]}")
    print(f"Remote file shape: {df_sent.shape}")

Local file last date: 3/29/2026
Local file shape: (16871, 2)
Remote file last date: New York Post 
Remote file shape: (27, 10)


## Summary

After running all cells above, we can determine:
- Whether each source is still accessible via direct download/API
- How recent the data is (how far behind today's date)
- What approach to take for live deployment